In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tkinter as tk
from tkinter import ttk
import joblib
import os

#  Load Dataset
csv_path = r"D:/Projects/all_fraud_detection/notebook/credit_combined_dataset.csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError(" Dataset not found at the specified path.")

df_raw = pd.read_csv(csv_path)
df_raw.columns = df_raw.columns.str.strip()  # Strip column whitespace

#  Verify required columns
required_columns = [
    'sequence_no', 'transaction_id', 'card_number', 'card_holder_name',
    'merchant_category', 'transaction_amount', 'transaction_date',
    'transaction_time', 'transaction_type', 'location', 'channel',
    'previous_fraud_count', 'is_international', 'label'
]
missing = [col for col in required_columns if col not in df_raw.columns]
if missing:
    raise ValueError(f" Missing columns: {missing}")

#  Preprocess Data
df = df_raw.dropna()
df['label'] = df['label'].str.strip().map({'not fraud': 0, 'fraud': 1})
if df['label'].isnull().any():
    raise ValueError(" 'label' column has invalid values.")

X = df.drop(['sequence_no', 'transaction_id', 'card_number', 'card_holder_name', 'label'], axis=1)
y = df['label']

#  Label Encode categorical columns
label_encoders = {}
for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        label_encoders[col] = le

#  Train Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
joblib.dump(model, "credit_fraud_model.pkl")

print(" Model trained and saved as 'credit_fraud_model.pkl'")

#  GUI Prediction Function
def predict_fraud(seq_no):
    try:
        seq_no = int(seq_no)
        row = df_raw[df_raw['sequence_no'] == seq_no]
        if row.empty:
            result_var.set(" Sequence No. not found")
            return

        row_data = row.iloc[0]
        for i, var in enumerate(field_vars):
            var.set(str(row_data[columns[i]]))

        #  Drop unused fields and encode
        input_row = row.drop(['sequence_no', 'transaction_id', 'card_number', 'card_holder_name', 'label'], axis=1).copy().iloc[0]

        for col in input_row.index:
            if col in label_encoders:
                input_row[col] = label_encoders[col].transform([input_row[col]])[0]

        prediction = model.predict([input_row.values])[0]
        result = " Fraud Detected" if prediction == 1 else "✅ Transaction is Safe"
        result_var.set(result)

    except Exception as e:
        result_var.set(f" Error: {e}")
        
#  GUI Setup
root = tk.Tk()
root.title("💳 Credit Card Fraud Detection System")
root.geometry("520x750")
root.configure(bg="white")

tk.Label(root, text="Enter Sequence Number:", bg="white", font=('Arial', 12)).pack(pady=10)
seq_var = tk.StringVar()
ttk.Entry(root, textvariable=seq_var).pack(pady=5)

tk.Button(root, text=" Predict", command=lambda: predict_fraud(seq_var.get()), bg="green", fg="white", font=('Arial', 12)).pack(pady=15)

#  Field display
columns = [
    'transaction_id', 'card_number', 'card_holder_name', 'merchant_category',
    'transaction_amount', 'transaction_date', 'transaction_time',
    'transaction_type', 'location', 'channel', 'previous_fraud_count', 'is_international'
]
field_vars = [tk.StringVar() for _ in columns]

for label, var in zip(columns, field_vars):
    tk.Label(root, text=label.replace('_', ' ').title() + ":", bg="white", font=('Arial', 10)).pack(fill='x', padx=20)
    tk.Entry(root, textvariable=var, state='readonly', bg="#f0f0f0", font=('Arial', 10)).pack(fill='x', padx=20, pady=2)

#  Result Output
result_var = tk.StringVar()
tk.Label(root, text="Prediction Result:", bg="white", font=('Arial', 14)).pack(pady=10)
tk.Label(root, textvariable=result_var, bg="white", font=('Arial', 16, 'bold'), fg="blue").pack(pady=5)

#  Start the GUI loop
root.mainloop()


 Model trained and saved as 'credit_fraud_model.pkl'


c:\Users\Sayal\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
